In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame
from listUtils import getFlatList
from musicdb import MusicDBIO
from sys import prefix
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM']
MasterParams()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb


# DB-Specific

In [4]:
from lib import setlistfm
mio   = setlistfm.MusicDBIO(verbose=False, mkDirs=False)
apiio = setlistfm.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant SetListFM DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/SetListFM


# Master Files

In [5]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = Series(dtype='object') #mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [6]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

SetListFM Search Results
   Global Master Search:      7147
   Global Master Search Data: 1439
   Search Artists:            5873
   Errors:                    6
   Known Summary IDs:         0


# Search For New Artists

In [7]:
mio   = setlistfm.MusicDBIO(verbose=False,local=False,mkDirs=False)
apiio = setlistfm.RawAPIData(debug=False)

## Find Artists To Download

In [13]:
from musicdb import MusicDBIO
from gate import MusicDBGate

pdbio = MusicDBIO()
mmeDF = pdbio.getData().sort_values(by=["Counts", "Albums"], ascending=False)

gate    = MusicDBGate()
mdbio   = gate.getIO(db="MusicBrainz")
refData = mdbio.data.getSummaryRefData().to_dict()

mbIDData = mmeDF[mmeDF["MusicBrainz"].notna()][["ArtistName", "MusicBrainz"]]
mbIDData["MBRef"] = mbIDData["MusicBrainz"].apply(refData.get).apply(lambda x: x.split('/')[-1] if isinstance(x,str) else None)

searchedForMasterArtists = masterArtists.get()
artistNamesToGet = mbIDData[~mbIDData["MusicBrainz"].isin(searchedForMasterArtists.keys())]

print("{0} Search Results".format(db))
print("   Known Master Artist Names:    {0}".format(mbIDData.shape[0]))
print("   Known Spotify Artist Names:   {0}".format(len(searchedForMasterArtists)))
print("   Artist Names To Get:          {0}".format(len(artistNamesToGet)))


#   Artist Names To Get:          804569
#   Artist Names To Get:          802284

SetListFM Search Results
   Known Master Artist Names:    810305
   Known Spotify Artist Names:   8007
   Artist Names To Get:          802284


## Download Artist Searches

In [14]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
#tt = TermTime("today", "7:00pm")
tt = TermTime("today", "11:50pm")
n    = 0
maxN = 1400

searchedForMasterArtistsData = masterArtistsData.get()
searchedForMasterArtists     = masterArtists.get()
searchedForErrors            = errors.get()
nErr = []
for i,(idx,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["ArtistName"]
    artistID = row["MusicBrainz"]
    mbID = row["MBRef"]
    if searchedForMasterArtists.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    response = apiio.getArtistInfoResults(artistName=artistName, mbID=mbID)
    if response is None:
        print("Error ==> {0}".format((artistID,mbID,artistName)))
        searchedForErrors[artistID] = True
        apiio.sleep(15)
        nErr.append(artistID)
        if len(nErr) >= 6:
            break
        continue

    nErr = []
    searchedForMasterArtistsData[artistID] = response
    searchedForMasterArtists[artistID] = artistName
    apiio.sleep(15)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(5.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForMasterArtists), db))
        masterArtists.save(data=searchedForMasterArtists)
        masterArtistsData.save(data=searchedForMasterArtistsData)
        errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(6.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break
            
ts.stop()
print("Saving [{0} / {1}] {2} Searched For Artist IDs".format(len(searchedForMasterArtists), len(searchedForMasterArtistsData), db))
masterArtists.save(data=searchedForMasterArtists)
masterArtistsData.save(data=searchedForMasterArtistsData)
if len(nErr) > 0:
    errors.save(data=searchedForErrors)
    for artistID in nErr:
        print("del searchedForErrors['{0}']".format(artistID))
    print("errors.save(data=searchedForErrors)")

Process [Getting SetListFM ArtistIDs] Start    ==> Time Is 2022-04-05 14:53:56
========================= termTime(day=today,time=11:50pm) =========================
   ====> Terminate Time Set To 2022-04-05 23:50:00 <====
   ====> Will Terminate Process 8 Hours and 56 Minutes From Now
Searching For Matisse & Sadko (c02a6ce9-452c-4df5-a065-4928e6bf07e6)                          

ERROR:repertorio.repertorio:HTTP exception: HTTPError('404 Client Error: Not Found for url: https://api.setlist.fm/rest/1.0/artist/c02a6ce9-452c-4df5-a065-4928e6bf07e6')


==> Error in SetListFM search for Matisse & Sadko
Error ==> ('93073986216613742411504369271603828645', 'c02a6ce9-452c-4df5-a065-4928e6bf07e6', 'Matisse & Sadko')
Searching For Domo Genesis (0ad99596-cbec-4726-af16-138e2a9268a1)                             True
Searching For Michael Christmas (0211a7f3-a769-4f36-aa8c-030e62ad6b18)                        True
Searching For Buckwheat Zydeco (ca2b920b-702a-4c4a-9e22-a07e052b4fd8)                         True
Searching For Yael Naïm (def226b0-7990-4b38-ab1a-7740461844dc)                               True
Searching For Les Fatals Picards (1a71713c-6cef-49e2-b2b7-8cdb6eb3f8cf)                       True
5/?        : Process [Getting SetListFM ArtistIDs] Has Run For 1 Minute and 44 Seconds.
Saving 8012 SetListFM Searched For Artist IDs
Searching For Keny Arkana (f97c4058-cc2b-43da-97e5-96bef7e55f40)                              True
Searching For Taste (e48d590f-1352-46a0-9da1-2ce4ec6196b5)                                    True
Searching F

Searching For Tad (867d3fb7-9cd1-47a4-a8f0-de29cdca2004)                                      True
Searching For Feint (e6e29ff5-b0e4-4985-a63f-e99d12b21728)                                    True
Searching For Trapeze (d025233d-5d24-4a41-81dd-98f64a058222)                                  True
Searching For Cupid (56e347d3-0306-4a32-a015-f9d19ebf56fb)                                    True
Searching For Lemon Jelly (4cbbf0d6-6c78-4e3d-ab25-91b13603cf7c)                              True
50/?       : Process [Getting SetListFM ArtistIDs] Has Run For 15 Minutes and 27 Seconds.
Saving 8057 SetListFM Searched For Artist IDs
Searching For Bruna Karla (f570efbe-5789-4e9b-ae09-9c5b53a9be92)                              True
Searching For Kimya Dawson (ba99a190-6065-4930-be3d-55ecc48e365d)                             True
Searching For Masterplan (bd496eba-155e-49ba-9a92-56e3fdfd1ea2)                               True
Searching For PeterLicht (56c2dfbb-6b0c-4a11-9673-67ff03e4b17d)         

Searching For David Ball (a0e567d7-0438-49c2-98af-c38f3ea50662)                               True
Searching For H-Town (e7bc282e-f651-4448-8716-2dd5f3b16c1d)                                   True
Searching For Big Big Train (395050f8-a66e-49eb-a915-0fd68253eb6f)                            True
Searching For Cate Le Bon (f2393e49-b791-46da-a6d7-1e9e60743405)                              True
Searching For The Civil Wars (91dad7e7-0bf6-47e8-bd42-ef1fac32c729)                           True
95/?       : Process [Getting SetListFM ArtistIDs] Has Run For 29 Minutes and 17 Seconds.
Saving 8102 SetListFM Searched For Artist IDs
Searching For Impetigo (22fe314d-f914-4745-9f79-eeeb9fb2c370)                                 True
Searching For LOUISAHHH!!! (70900801-c182-4906-b870-4bf8e2b8129b)                             True
Searching For The Eli Young Band (80737f69-60c5-4c0d-9562-88faf8d8f2f4)                       True
Searching For Jala Brat (b83cc375-fd60-49ad-a26f-0c02bec189a5)          

...

==> Error in SetListFM search for Marwa Loud
Error ==> ('128484191260245166111627934162640762527', '2d3241e8-ef2e-4fa4-840c-2d362dcd151c', 'Marwa Loud')
Searching For MC Frontalot (abc7cbc7-a3bc-449a-867e-f8b22dded93c)                             

ERROR:repertorio.repertorio:HTTP exception: HTTPError('429 Client Error: Too Many Requests for url: https://api.setlist.fm/rest/1.0/artist/abc7cbc7-a3bc-449a-867e-f8b22dded93c')


==> Error in SetListFM search for MC Frontalot
Error ==> ('71593637436551729881680254667595741744', 'abc7cbc7-a3bc-449a-867e-f8b22dded93c', 'MC Frontalot')
Searching For Kae Tempest (7a2533c3-790e-4828-9b30-ca5467c609c5)                              

ERROR:repertorio.repertorio:HTTP exception: HTTPError('429 Client Error: Too Many Requests for url: https://api.setlist.fm/rest/1.0/artist/7a2533c3-790e-4828-9b30-ca5467c609c5')


==> Error in SetListFM search for Kae Tempest
Error ==> ('296309711276581535718979428770152103565', '7a2533c3-790e-4828-9b30-ca5467c609c5', 'Kae Tempest')
Process [Getting SetListFM ArtistIDs] Ran For 2 Hours and 56 Minutes    ==> Time Is 2022-04-05 17:50:30
Saving [8583 / 576] SetListFM Searched For Artist IDs
del searchedForErrors['54907945736023855028666453233518952507']
del searchedForErrors['254228023097287349139890783288984474288']
del searchedForErrors['80638289028284692601617462456405741646']
del searchedForErrors['128484191260245166111627934162640762527']
del searchedForErrors['71593637436551729881680254667595741744']
del searchedForErrors['296309711276581535718979428770152103565']
errors.save(data=searchedForErrors)


In [ ]:
del searchedForErrors['138309139701422690155865074761089162054']
del searchedForErrors['177142020761047431409012641832778216588']
del searchedForErrors['337416445052122721387209166204088433345']
del searchedForErrors['304209454874728599137188574098635691038']
del searchedForErrors['188621051952302411690967785989590288245']
del searchedForErrors['190341879716129587744546996111542622292']
errors.save(data=searchedForErrors)

## Save Results

In [10]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        df = Series(mad).apply(Series)
        #df = DataFrame({v['mbid']: {k2: v2 for k2,v2 in v.items() if k2 not in []} for k,v in mad.items()}).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

ERROR! Session/line number was not unique in database. History logging moved to new session 6029


In [11]:
mad = masterArtistsData.get()
df  = getResponseDataFrame(mad)

if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = setlistfm.MusicDBIO(local=False).data.getSearchArtistData()
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF.drop_duplicates(keep='first')
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("Saving Data")
    setlistfm.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)

Found 2299 New Artists
Found 5873 Previous Artists
Found 8172 Total Results
Found 8007 Unique Results
Saving Data


In [12]:
masterArtistsData.save(data={})

# Download Artist Data

In [ ]:
mio   = metalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = metalarchives.RawAPIData(debug=False)

## Find Artists To Download

In [ ]:
artistData = {}
for searchTerm,searchResults in searchArtists.iteritems():
    if isinstance(searchResults,list):
        artistData.update({item["id"]: item for item in searchResults if isinstance(item,dict)})
artistData       = DataFrame(artistData).T.sort_values(by='id')
artistNames      = artistData[["name", "url"]]
localArtistsDict = localArtists.get()
artistIDsToGet   = artistNames[~artistNames.index.isin(localArtistsDict.keys())].sample(frac=1)

print("{0} Search Results".format(db))
print("   Available IDs:      {0}".format(len(artistNames)))
print("   Known Artist IDs:   {0}".format(len(localArtistsDict)))
print("   Artist IDs To Get:  {0}".format(len(artistIDsToGet)))

## Download The Data

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "12:05pm")
maxN = 50000000

n  = 0
localArtistsDict     = localArtists.get()
searchedForErrors    = errors.get()
for i,(artistID,row) in enumerate(artistIDsToGet.iterrows()):
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    artistName = row["name"]
    artistURL  = row["url"]

    dbID   = artistID
    modVal = mio.mv.get(dbID)
    if mio.data.getRawArtistInfoFilename(modVal, dbID).exists():
        localArtistsDict[artistID] = artistName
        continue
    try:
        response = apiio.getArtistInfoResults(artistName=artistName, artistURL=artistURL)
    except:
        response = None
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistID] = True
        apiio.sleep(3.5)
        continue
    
    localArtistsDict[artistID] = artistName
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=dbID)
    apiio.sleep(6.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

In [ ]:
ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

In [ ]:
localArtists.save(data=localArtistsDict)

In [ ]:
mio.data.getRawArtistInfoData(mio.mv.get(3540473060), 3540473060)

In [ ]:
localArtistsDict

In [ ]:
localArtists.save(data=localArtistsDict)


In [ ]:
tracks
disc_count